# Input Channel Disclosure Table + AgentConn Downstream Supplementary Evaluation — Read-Only Presentation

> **Read-only notebook**: this notebook only loads and displays the artifacts under `results/exp_r214/`; it performs no computation or file writes itself.
> Artifacts are generated by `029_eval_agentconn_downstream.py` (the disclosure table is machine-generated via three independent paths — importlib inspection, static source-code scanning, and graph-cache measurement; the downstream supplementary evaluation runs CPU inference plus Voronoi aggregation over the 48 frozen AgentConn models).

## What this experiment addresses

The paper needs to explicitly disclose the model's **input channels** — what the GNN actually "sees": which signals enter the feature vector, which only pass through the loss, what edges exist in the graph, and what the softmax temperature is. This artifact reads these facts directly from the source (module attributes / source-code line numbers / cache measurements), each item carrying a `source` provenance field for direct citation in the methods section's disclosure table; `tests/test_r214.py` asserts against the artifact fields to guard against manual transcription errors.

This also addresses the question of the missing agent-agent edges: the existing ablation (`agent_conn_ablation_report.md`) only has training-time val-loss evidence (showing no difference); this experiment fills in the comparison at the downstream allocation-quality level (RMSE/MAE/Corr) — AgentConn baseline vs. the main baseline, using the established statistical protocol (seed averaging → 16 paired regional differences → exact 2^16 sign-flip permutation test).

In [ ]:
# Read-only: load artifacts
import json
from pathlib import Path

import pandas as pd

EXP_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = EXP_DIR / 'results' / 'exp_r214'

with open(OUT / 'input_channel_disclosure.json', encoding='utf-8') as f:
    D = json.load(f)

print('Generated at:', D['meta']['generated_at'])
print('Generation methods:')
for gm in D['meta']['generation_methods']:
    print('  -', gm)

## 1. Input Channel Disclosure Table (value + source per item)

Draft source for the paper's methods-section disclosure table. **Key facts**: the agent feature vector = 5-dimensional land-use proportions (no NTL, no coordinates); NTL / Proximity enter the training gradient only through the prior loss (not used at all at inference time); edges = ITL3 membership star topology (each agent connects to exactly one source); the main experiment uses `AGENT_CONNECTIVITY=None` (no agent-agent edges); τ is initialized to 0.01 (2 occurrences in script 005, consistent value).

In [ ]:
items = D['items']

rows = []
for name, item in items.items():
    v = item['value']
    if name in ('main_graph_cache', 'agentconn_graph_cache'):
        summary = (f"nodes={v['node_types']} | edges={['-'.join(et) for et in v['edge_types']]} | "
                   f"agent feature dim={v['agent_feature_dim']} | total near edges={v['near_edge_total']} | "
                   f"star membership={v['star_membership_structure_all_graphs']}")
    elif name == 'config_map_losses':
        summary = ' ; '.join(f"{k}:{'+'.join(vv['losses'])}(ep{vv['epochs']})"
                             for k, vv in v.items())
    elif name == 'tau_initial':
        summary = (f"occurs {v['n_occurrences']} times "
                   f"(L{','.join(str(o['line']) for o in v['occurrences'])}), "
                   f"consistent={v['all_equal']}, value={v['unique_value']}")
    elif name == 'agent_feature_cols':
        summary = (f"{v['count']} columns: {v['columns']}"
                   f" (contains NTL={v['contains_ntl']}, contains coordinates={v['contains_coordinates']})")
    elif isinstance(v, dict) and 'meaning' in v:
        summary = v['meaning']
    else:
        summary = json.dumps(v, ensure_ascii=False)
    rows.append({'Item': name, 'Fact (value summary)': summary, 'source (provenance)': item['source']})

with pd.option_context('display.max_colwidth', 110):
    display(pd.DataFrame(rows))

### 1.1 Channel Comparison Summary (rule-generated from the disclosure-table fields)

The table below summarizes "signal → entry path" (rendered directly from the fields above, not hand-written):

In [ ]:
fc = items['agent_feature_cols']['value']
ntl = items['ntl_pathway']['value']
prox = items['proximity_pathway']['value']
mg = items['main_graph_cache']['value']
ag = items['agentconn_graph_cache']['value']

channel_rows = [
    {'Signal': 'Land-use proportions (5-dim)', 'Entry path': 'Node feature agent.x (the only feature channel)',
     'Evidence': f"AGENT_FEATURE_COLS={fc['columns']}"},
    {'Signal': 'Coordinates (geometry)',
     'Entry path': 'Not included in features — dropped before graph construction; retained only as a separate coords attribute (not consumed by the forward pass)',
     'Evidence': items['graphbuilder_geometry_drop']['value']['line_text']},
    {'Signal': 'NTL nighttime lights',
     'Entry path': f"Only through the {ntl['loss_registry_key']} loss (configs {ntl['configs_with_this_prior']})",
     'Evidence': f"in_agent_feature_cols={ntl['in_agent_feature_cols']}"},
    {'Signal': 'Proximity',
     'Entry path': f"Only through the {prox['loss_registry_key']} loss (configs {prox['configs_with_this_prior']})",
     'Evidence': f"in_agent_feature_cols={prox['in_agent_feature_cols']}"},
    {'Signal': 'Graph topology',
     'Entry path': 'ITL3 membership star topology (source↔agent, bidirectional); no agent-agent edges in the main experiment',
     'Evidence': f"main cache near-edges={mg['near_edge_total']}, AgentConn cache near-edges={ag['near_edge_total']}"},
]
with pd.option_context('display.max_colwidth', 100):
    display(pd.DataFrame(channel_rows))

## 2. AgentConn Downstream Supplementary Evaluation: Per-Region Comparison Against the Main Baseline

The 48 frozen AgentConn models (HPC-trained, including agent-agent `near` edges) run CPU inference on their own graph_cache, using the same seven demand columns as script 005, plus Voronoi aggregation, producing the same three `kfold_test` metrics as exp0 (Voronoi convention only; CIVD is skipped here, consistent with established practice).

In [ ]:
import numpy as np

DS = OUT / 'agentconn_downstream'
if not (DS / 'agentconn_vs_main.csv').exists():
    print('AgentConn downstream artifacts not generated (optional step not run, or downgraded to disclosure-table-only)')
else:
    vs = pd.read_csv(DS / 'agentconn_vs_main.csv')
    with open(DS / 'run_meta.json', encoding='utf-8') as f:
        meta = json.load(f)
    print(f"device={meta['device']} | failed models={meta['n_models_failed']}/{meta['n_models_expected']} "
          f"| splits verified={meta['splits_verified_against_exp0']} | elapsed={meta['elapsed_seconds']}s")

    # 16-region mean comparison per config × metric (voronoi_GNN arm)
    pivot = (vs[vs['method'] == 'voronoi_GNN']
             .groupby(['config', 'metric'])[['agentconn_mean', 'main_mean', 'diff_mean']]
             .mean().round(4))
    display(pivot)

## 3. Statistical Test: AgentConn Baseline vs. Main Baseline (voronoi_GNN)

Protocol = seed averaging → 16 paired regional differences → exact 2^16 sign-flip permutation test; CI = region-level paired bootstrap (outer level only, B=10⁴); the 3 per-seed p-values are reported as-is and combined via the Cauchy combination method (the median is not used for combining). diff = AgentConn − main.

In [ ]:
if (DS / 'agentconn_vs_main_test.json').exists():
    with open(DS / 'agentconn_vs_main_test.json', encoding='utf-8') as f:
        T = json.load(f)
    trows = []
    for metric, c in T['comparisons'].items():
        if 'mean_diff' not in c:
            trows.append({'Metric': metric, 'Status': c.get('status', 'skipped')})
            continue
        trows.append({
            'Metric': metric,
            'mean_diff (AC−main)': round(c['mean_diff'], 4),
            'perm_p (exact 2^16)': c['perm_p'],
            '95% CI': f"[{c['ci_lo']:.4f}, {c['ci_hi']:.4f}]",
            'per-seed p': json.dumps(c['per_seed_perm_p']),
            'Cauchy-combined p': round(c['cauchy_combined_p'], 4),
            'Significant@0.05': c['significant_at_0.05'],
        })
    display(pd.DataFrame(trows))
else:
    print('Test JSON not generated')

## 4. Conclusions (generated programmatically from the numeric results)

In [ ]:
# All conclusion lines are generated programmatically from the saved numbers; none are hand-written
if (DS / 'agentconn_vs_main_test.json').exists():
    con = T['conclusion']
    if 'direction' in con:
        print('Direction:      ', con['direction'])
        print(f"RMSE difference:  {con['rmse_mean_diff']:+.4f}  (perm p = {con['rmse_perm_p']:.4f})")
        print('Any metric significant@0.05:', con['any_metric_significant_at_0.05'])
        print('Consistent with val-loss no-difference conclusion:', con['consistent_with_val_loss_no_difference'])
        print('Paper action:  ', con['paper_action'], '——', con['paper_action_rule'])
    else:
        print('Test downgraded:', con)
else:
    print('AgentConn downstream not run; only the disclosure table is available in this section')